# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/brandi/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/brandi/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/brandi/student/ai_makerspace/ai-makerspace-course/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/brandi/student/ai_makerspace/ai-makerspace-course/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/brandi/student/ai_makerspace/ai-makerspace-course/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 40, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/34 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/40 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/62 [00:00<?, ?it/s]

Property 'summary' already exists in node 'b74f19'. Skipping!
Property 'summary' already exists in node 'bf41f8'. Skipping!
Property 'summary' already exists in node '75dc5a'. Skipping!
Property 'summary' already exists in node '726867'. Skipping!
Property 'summary' already exists in node '764108'. Skipping!
Property 'summary' already exists in node 'eeaec3'. Skipping!
Property 'summary' already exists in node '20b93c'. Skipping!
Property 'summary' already exists in node '0e619d'. Skipping!
Property 'summary' already exists in node '99c855'. Skipping!
Property 'summary' already exists in node '6e0db9'. Skipping!
Property 'summary' already exists in node 'a73fae'. Skipping!
Property 'summary' already exists in node '566aab'. Skipping!
Property 'summary' already exists in node '842388'. Skipping!
Property 'summary' already exists in node '13be37'. Skipping!
Property 'summary' already exists in node '84d1bb'. Skipping!
Property 'summary' already exists in node '0d07c3'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/86 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'b74f19'. Skipping!
Property 'summary_embedding' already exists in node '06112b'. Skipping!
Property 'summary_embedding' already exists in node '6e0db9'. Skipping!
Property 'summary_embedding' already exists in node '726867'. Skipping!
Property 'summary_embedding' already exists in node '99c855'. Skipping!
Property 'summary_embedding' already exists in node '75dc5a'. Skipping!
Property 'summary_embedding' already exists in node 'bf41f8'. Skipping!
Property 'summary_embedding' already exists in node '20b93c'. Skipping!
Property 'summary_embedding' already exists in node 'eeaec3'. Skipping!
Property 'summary_embedding' already exists in node 'a73fae'. Skipping!
Property 'summary_embedding' already exists in node '842388'. Skipping!
Property 'summary_embedding' already exists in node '764108'. Skipping!
Property 'summary_embedding' already exists in node '0e619d'. Skipping!
Property 'summary_embedding' already exists in node '566aab'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 80, relationships: 1931)

We can save and load our knowledge graphs as follows.

In [11]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 80, relationships: 1931)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

- `SingleHopSpecificQuerySynthesizer`: This synthesizer is the most simple, creating "single hop" queries. In a nutshell, that means it's creating user queries that can be answered in full by a **single** retrieved document. The `Specific` piece means that we're looking at concrete knowledge or facts. Something like "When did world war 2 start?" not "Why did world war 2 start?"

- `MultiHopAbstractQuerySynthesizer`: As the name implies, this synthesizer is creating user queries that require information that spans across more than one document. If our retriever fails to fetch the multiple documents, it will be unable to answer this query. The `Abstract` piece means that we're asking more conceptual questions that require reasoning or abstract thinking to answer. Something like "How do different types of federal aid work together to support a student’s education costs?" 

- `MultiHopSpecificQuerySynthesizer`: This is another multi-hop synthesizer, meaning that we need more than one document, or more than one "hop" to answer a question. The "specific" means we're looking for tangible, factual information. Something like "What steps must a parent take to apply for a Direct PLUS Loan after submitting the FAFSA?"



Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are the defintions of Academic Years in t...,"[Chapter 1 Academic Years, Academic Calendars,...","Chapter 1 Academic Years, Academic Calendars, ...",single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(b) about in the context o...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums ar...,single_hop_specifc_query_synthesizer
2,Could you explain the significance of Volume 8...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,What is the Federal Work-Study program and how...,[Non-Term Characteristics A program that measu...,The Federal Work-Study (FWS) Program is an exc...,single_hop_specifc_query_synthesizer
4,What is Volume 7 about?,[both the credit or clock hours and the weeks ...,Volume 7 provides guidance on the disbursement...,single_hop_specifc_query_synthesizer
5,How does credit hour allocation for clinical e...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in a standard t...,multi_hop_abstract_query_synthesizer
6,How do the timing and scheduling constraints o...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_abstract_query_synthesizer
7,clinical work timing overlapping courses how d...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_abstract_query_synthesizer
8,Based on the regulatory citations 34 CFR 668.3...,[<1-hop>\n\nRegulatory Citations Academic year...,The regulatory citation 34 CFR 668.3(a) specif...,multi_hop_specific_query_synthesizer
9,How do the definitions of academic years in Vo...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 outlines the general requirements for...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '03a7b6'. Skipping!
Property 'summary' already exists in node 'b19664'. Skipping!
Property 'summary' already exists in node 'df8995'. Skipping!
Property 'summary' already exists in node 'eb6f0f'. Skipping!
Property 'summary' already exists in node '3ffd71'. Skipping!
Property 'summary' already exists in node '2b5b5c'. Skipping!
Property 'summary' already exists in node '6951b8'. Skipping!
Property 'summary' already exists in node '422642'. Skipping!
Property 'summary' already exists in node '0ab14e'. Skipping!
Property 'summary' already exists in node '3b1087'. Skipping!
Property 'summary' already exists in node '3d515e'. Skipping!
Property 'summary' already exists in node '8f0ce1'. Skipping!
Property 'summary' already exists in node '32db10'. Skipping!
Property 'summary' already exists in node '7bf1d8'. Skipping!


...

Property 'summary_embedding' already exists in node '03a7b6'. Skipping!
Property 'summary_embedding' already exists in node 'b19664'. Skipping!
Property 'summary_embedding' already exists in node '6951b8'. Skipping!
Property 'summary_embedding' already exists in node 'df8995'. Skipping!
Property 'summary_embedding' already exists in node '2b5b5c'. Skipping!
Property 'summary_embedding' already exists in node '8f0ce1'. Skipping!
Property 'summary_embedding' already exists in node 'eb6f0f'. Skipping!
Property 'summary_embedding' already exists in node '3b1087'. Skipping!
Property 'summary_embedding' already exists in node '3ffd71'. Skipping!
Property 'summary_embedding' already exists in node '0ab14e'. Skipping!
Property 'summary_embedding' already exists in node '7bf1d8'. Skipping!
Property 'summary_embedding' already exists in node '422642'. Skipping!
Property 'summary_embedding' already exists in node '3d515e'. Skipping!
Property 'summary_embedding' already exists in node '32db10'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [21]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does the FAFSA process incorporate the FAF...,[Application and Verification Guide Introducti...,The FAFSA process has been overhauled by the F...,single_hop_specifc_query_synthesizer
1,Can you tell me about the FAFSA and what happe...,[Chapter 1: The Application Process We removed...,Chapter 1 states that the Returning FAFSA File...,single_hop_specifc_query_synthesizer
2,What is Federal Tax Information used for in FA...,[electronically due to limitations on access t...,The context does not explicitly explain what F...,single_hop_specifc_query_synthesizer
3,What is the link to the 2024-25 FAFSA Partner ...,[Note: There will be two active portals that c...,The link to the 2024-25 FAFSA Partner Portal i...,single_hop_specifc_query_synthesizer
4,how returning FAFSA filers do they need to rea...,[<1-hop>\n\nApplication and Verification Guide...,The context indicates that the section on Retu...,multi_hop_abstract_query_synthesizer
5,How does the removal of COVID-19 related guida...,[<1-hop>\n\nApplication and Verification Guide...,The removal of COVID-19 related guidance has l...,multi_hop_abstract_query_synthesizer
6,How does the use of IRS data for FAFSA eligibi...,[<1-hop>\n\nApplication and Verification Guide...,The use of IRS data for FAFSA eligibility dete...,multi_hop_abstract_query_synthesizer
7,How do restrictions on FAFAs initiating new ap...,[<1-hop>\n\nelectronically due to limitations ...,Restrictions on FAFAs initiating new applicati...,multi_hop_abstract_query_synthesizer
8,"In chapter 2, student family size example and ...",[<1-hop>\n\nStudent Citizenship (13) Family Si...,"According to Chapter 2, the student can includ...",multi_hop_specific_query_synthesizer
9,How do Chapters 2 and 5 relate to determining ...,[<1-hop>\n\nStudent Citizenship (13) Family Si...,Chapter 2 provides detailed examples of how fa...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context include:\n\n- Direct Subsidized Loans (available only to undergraduate students)\n- Direct Unsubsidized Loans\n- Direct PLUS Loans (student Federal PLUS Loans and parent PLUS Loans)\n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)\n- Federal SLS Loans (under FFEL Program before July 1, 2010)\n- Federal PLUS Loans (under FFEL Program before July 1, 2010)\n\nAdditionally, there are Direct Consolidation Loans and Federal Consolidation Loans mentioned, which consolidate previous loans.\n\nTherefore, the available loans include Direct Subsidized Loans, Direct Unsubsidized Loans, Direct PLUS Loans, and loans made previously under the FFEL Program such as Federal Stafford Loans, Federal SLS Loans, and Federal PLUS Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

##### ✅ Answer:

- `qa_evaluator`: This is the most basic "Is this correct" evaluator. Is it factually accurate?
- `labeled_helpfulness_evaluator`: This evaluator is seeing if the answer is helpful. It's possible to have a correct answer without being helpful. ("How do I cook a pizza?" "In the oven." True, not helpful.)
- `empathy_evaluator`: This evaluator is checking the empathy of responses to users. Given the topic, it's important to be sensitive and not overly-blunt. This evaluator will focus on tone and empathy rather than factuality or helpfulness.

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'respectful-crown-56' at:
https://smith.langchain.com/o/17336763-e025-4cab-8ce1-f61b7408e302/datasets/c4ba55b1-2ba6-4205-babb-4da52468a433/compare?selectedSessions=0616ba5c-9938-4622-b968-e2d0c02bed14




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Volume 8 and Volume 8 how does that affect the...,I don't know.,None,"According to Volume 8, the timing of disbursem...",0,0,0,1.732841,ed261832-0ca1-4aed-8251-6894c7eed54f,8fb7977a-0a7b-4924-a109-84fc6d55bf77
1,How does Volume 8 address disbursement timing ...,"Based on the provided context, Volume 8 addres...",None,Volume 8 explains that for subscription-based ...,0,0,0,3.327087,21262845-4614-42db-a01b-316242279af3,498d70bd-ee70-4708-956e-4ff51a2bfc8c
2,How do Appendix A and Appendix B relate to dis...,"Based on the provided context, Appendix B spec...",None,Appendix B provides detailed guidance on disbu...,0,0,0,4.243987,45efff24-da11-4b35-88fc-ba4da94127d3,b61b1822-f9da-43df-8c55-ca132d70a1c3
3,"In volume 2 and volume 7, how does the academi...",Based on the provided context:\n\n- The defini...,None,"In volume 2, it explains that the academic yea...",1,1,0,8.133236,cd31e508-4823-46df-867a-ed08ba84843b,96a484d8-2f28-40f1-87bf-cdd48a290b75
4,Hwo do the academic years and regulatory citat...,"Based on the provided context, academic years ...",None,"The academic years, which must include a minim...",1,1,0,5.069991,dc955691-6d07-4942-9055-1426785bf6e5,13a21814-3db7-4c8a-889e-504a581c8c7c
5,Hwo does the academic year based on instructio...,"Based on the provided context, the academic ye...",None,The academic year based on instructional time ...,1,1,0,4.606120,3a7a5f82-8e68-4fc1-a73c-b4ef32fab79f,0a29ca3f-3997-435a-8dab-2af92ec2aa21
6,What are the regulatory citations related to a...,The regulatory citations related to academic y...,None,The regulatory citations related to academic y...,1,1,0,2.970262,63393740-24fb-4180-9f68-e67dc30ee126,ed0a9107-366e-4fca-af1d-63669da5d1f0
7,How do nonstandard terms and payment periods r...,"Based on the provided context, nonstandard ter...",None,Nonstandard terms are defined as terms that do...,1,1,0,11.180976,59a23ed1-d351-406e-bbe7-6ba9238b99ac,c14ba036-9a4a-4097-8849-a93a4385bac5
8,How does the Federal Work-Study program differ...,The Federal Work-Study (FWS) program differs f...,None,The payment period is applicable to all Title ...,1,1,0,3.764071,25b73acd-17e5-4b55-956a-ff05e7cdc26c,f082d645-2a78-455e-a877-8f759027720e
9,Whaat is the defnition of a Standerd Term in t...,A standard term in the context of academic pro...,None,Inclusion of clinical work in a standard term ...,1,1,0,3.735574,1060d23f-bf8d-4552-9c7b-25c11d97916f,8aca0ed1-0fbe-4efc-8cb2-f809628c1614


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

The ideal chunk size varies based on your application, including factors like what model you're using, the format of your data, and the content of your data. If our chunks are too small, we're losing important context that is now spread across multiple chunks and now harder to reason about. If our chunks are too large, we might now be able to retrieve enough relevant data or fill up too much of our context window will irrelevant data.

Changing our chunk size can modify performance for these reasons.

In [35]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Changing our embedding model to a more powerful/larger model can increase the granularity in our understanding of chunk similarity. A better model will give us more accurate representations of relationships and similarities between chunks, leading to better and more relevant retrieval. Even if we were to change to an equally powerful embedding model, but one that was trained on data closer to our application's domain, we'll have better results. We may also be simply increasing the size of our embeddings, and therefore having more information about our chunks represented. In summary, there are many ways that changing our embedding model may affect the performance of our application!

In [36]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [37]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [38]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [39]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question! Based on the information in the context, several types of loans are available to help students and their families cover education costs:\n\n1. **Direct Subsidized Loans** – These are loans based on the student's financial need, and the government pays the interest while the student is in school at least half-time.\n\n2. **Direct Unsubsidized Loans** – These loans are available regardless of financial need, and interest accrues while the student is in school.\n\n3. **Direct PLUS Loans** – These loans are for graduate/professional students or the parents of dependent undergraduate students. Parents can borrow up to the student's Cost of Attendance (COA) minus other financial aid. However, there are eligibility requirements, and amounts cannot exceed COA less other aid.\n\nAdditionally, students may qualify for combinations of these loans, and if a dependent student's parent cannot obtain a Direct PLUS Loan, the student may be eligible for additional Direct U

Finally, we can evaluate the new chain on the same test set!

In [40]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'glossy-star-2' at:
https://smith.langchain.com/o/17336763-e025-4cab-8ce1-f61b7408e302/datasets/c4ba55b1-2ba6-4205-babb-4da52468a433/compare?selectedSessions=b789368a-eebb-40d6-9c59-0acaca04b0ca




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Volume 8 and Volume 8 how does that affect the...,Thank you for your thoughtful question. From t...,None,"According to Volume 8, the timing of disbursem...",1,0,1,8.803055,ed261832-0ca1-4aed-8251-6894c7eed54f,b247e6e0-0393-4979-86a8-351d9f692d62
1,How does Volume 8 address disbursement timing ...,Thank you for your thoughtful question. Based ...,None,Volume 8 explains that for subscription-based ...,0,0,1,6.617235,21262845-4614-42db-a01b-316242279af3,4da5d70c-8c49-48bd-9cb7-7f134218a506
2,How do Appendix A and Appendix B relate to dis...,Thank you for your question! Based on the cont...,None,Appendix B provides detailed guidance on disbu...,0,0,1,6.362182,45efff24-da11-4b35-88fc-ba4da94127d3,97c742b0-cf8f-4eac-b1a9-10af2ff71562
3,"In volume 2 and volume 7, how does the academi...",Thank you for your thoughtful question. Based ...,None,"In volume 2, it explains that the academic yea...",1,1,1,13.253356,cd31e508-4823-46df-867a-ed08ba84843b,ec13f137-226b-4927-b9df-92a0d7137463
4,Hwo do the academic years and regulatory citat...,Thank you for your thoughtful question—it's cl...,None,"The academic years, which must include a minim...",1,1,1,7.532269,dc955691-6d07-4942-9055-1426785bf6e5,1ad5e2e0-3c9b-493b-b83d-31aa813f8724
5,Hwo does the academic year based on instructio...,Thank you for your thoughtful question. Based ...,None,The academic year based on instructional time ...,0,0,1,4.926352,3a7a5f82-8e68-4fc1-a73c-b4ef32fab79f,cce0227e-5565-4c16-a91a-f96718401cbc
6,What are the regulatory citations related to a...,Thank you for your question. Based on the cont...,None,The regulatory citations related to academic y...,1,1,1,3.708011,63393740-24fb-4180-9f68-e67dc30ee126,c6f8b87b-61b9-44d0-971d-dac60d2b9152
7,How do nonstandard terms and payment periods r...,Thank you for your thoughtful question. From t...,None,Nonstandard terms are defined as terms that do...,1,1,1,7.138482,59a23ed1-d351-406e-bbe7-6ba9238b99ac,b9f0d355-e9e2-48a9-909f-2e89cfc582d6
8,How does the Federal Work-Study program differ...,Thank you for your thoughtful question. Based ...,None,The payment period is applicable to all Title ...,1,1,1,8.597369,25b73acd-17e5-4b55-956a-ff05e7cdc26c,d1914b89-7011-490b-a7e9-ba2c8f7e7d8c
9,Whaat is the defnition of a Standerd Term in t...,Thank you for your question! Based on the cont...,None,Inclusion of clinical work in a standard term ...,1,1,1,6.483931,1060d23f-bf8d-4552-9c7b-25c11d97916f,b8d8d608-169e-408c-bfa4-c1b0bfd0cda9


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

![Evals](eval_screenshot.png)

There are three things we can note based on these evals. For reference, `respectful-crown-56` was the first eval ran, and `glossy-star-2` was second after updates made to the application.

1. **Correctness stayed the same**: Our correctness evaluator resulted in the exact same percentage of answers being marked as correct. Given this, I would have a hard time justifying switching embedding models, or even chunk size. That said, we made several changes at once, so I might evaluate a single change at a time. This is also a very small evaluation, so trying out a larger dataset would likely give us a better idea of the difference in correctness, if any.

2. **Empathy**: This metric took a 180. On the first run, not a single response was marked as empathetic. In the second run, every single response was marked as empathetic. This is almost certainly 100% because of our prompt update that simply told the model to use empathy in its answers. When we provided a dry, fact-oriented prompt, we got back dry, fact-oriented responses. Adding this note about empathy made a huge difference in phrases like "I understand" and "I'm here to help".

3. **Helpfulness**: In the second run, two answers fewer were marked as helpful than in the first run. I would say that we don't have enough information to know why this change happened, or if it was even caused by a change or just something we're noticing because we have such a small dataset.